In [1]:
import json
import datetime
import time
import urllib 

from openai import AzureOpenAI
from azure.core.exceptions import AzureError
from azure.core.credentials import AzureKeyCredential

#Cosmos DB imports
from azure.cosmos import CosmosClient
from azure.cosmos.aio import CosmosClient as CosmosAsyncClient
from azure.cosmos import PartitionKey, exceptions

import configparser; config = configparser.RawConfigParser()
config.read('keys//keys.ini')

OPENAI_API_VERSION = "2023-05-15"
OPENAI_API_KEY = config['keys']['AzurekeyEUS']
OPENAI_API_ENDPOINT = """https://eastusOpenAIral.openai.azure.com/"""
EMBEDDING_MODEL_DEPLOYMENT_NAME = "textembeddings3large"
COSMOSDB_NOSQL_ACCOUNT_KEY = config['keys']['AzureCosmoDBcdbralvecKey']
COSMOSDB_NOSQL_ACCOUNT_ENDPOINT = config['keys']['COSMOSDB_ENDPOINT']


In [2]:
AOAI_client = AzureOpenAI(api_key=OPENAI_API_KEY, azure_endpoint=OPENAI_API_ENDPOINT, api_version=OPENAI_API_VERSION,)

In [3]:
def generate_embeddings(text):
    '''
    Generate embeddings from string of text.
    This will be used to vectorize data and user input for interactions with Azure OpenAI.
    '''
    response = AOAI_client.embeddings.create(input=text, model=EMBEDDING_MODEL_DEPLOYMENT_NAME)
    embeddings =response.model_dump()
    time.sleep(0.5) 
    return embeddings['data'][0]['embedding']

In [4]:
from azure.identity import DefaultAzureCredential
from azure.identity import ClientSecretCredential, DefaultAzureCredential

credential = DefaultAzureCredential()
cosmos_client = CosmosClient(COSMOSDB_NOSQL_ACCOUNT_ENDPOINT, credential, consistency_level="Strong")

In [5]:
# #create database
# DATABASE_NAME = "WP-vector-nosql-db-DiskANN"
# db= cosmos_client.create_database_if_not_exists(
#     id=DATABASE_NAME
# )
# properties = db.read()
# print(json.dumps(properties))


# vector_embedding_policy = {
#     "vectorEmbeddings": [
#         {
#             "path":"/contentVector",
#             "dataType":"float32",
#             "distanceFunction":"cosine",
#             "dimensions":3072
#         }
#     ]
# }

# #Add vector indexes to indexing policy
# indexing_policy  = {
#     "includedPaths": [
#         {
#             "path": "/*"
#         }
#     ],
#     "excludedPaths": [
#         {
#             "path": "/\"_etag\"/?"
#         }
#     ],
#     "vectorIndexes": [
#         {"path": "/contentVector",
#          "type": "diskANN"
#         }
#     ]
# }

# CONTAINER_NAME = "WP-vector-nosql-cont"
# try:   
#     container = db.create_container_if_not_exists(
#                     id=CONTAINER_NAME,
#                     partition_key=PartitionKey(path='/id', kind='Hash'),
#                     indexing_policy=indexing_policy,
#                     vector_embedding_policy=vector_embedding_policy)

#     print('Container with id \'{0}\' created'.format(id))

# except exceptions.CosmosResourceExistsError:
#     print('A container with id \'{0}\' already exists'.format(id))


# data = pd.read_csv("data\\WaP_with_embeddings.csv")
# data['category'] = 'Web'
# data.rename(columns={ 'paragraph' : 'content', 'embedding' : 'contentVector'}, inplace=True)
# data['id'] = data['id'].astype(str)
# datajson = data[['id', 'category', 'content', 'contentVector']].to_json(orient="records")
# datajson = json.loads(datajson)

# for i in range(len(datajson)):
#     datajson[i]['contentVector'] = json.loads(datajson[i]['contentVector'])

# container_client = db.get_container_client(CONTAINER_NAME)

# for item in datajson:
#     print("writing item ", item['id'])
#     container_client.upsert_item(item)




In [6]:
from azure.identity import DefaultAzureCredential
from azure.identity import ClientSecretCredential, DefaultAzureCredential

#az cosmosdb sql role assignment create -a cdbralvec -g cosmosdb -s "/" -p b85c6a4b-9678-4c00-9131-fc84b62d7d5d -d 00000000-0000-0000-0000-000000000002

credential = DefaultAzureCredential()
cosmos_client = CosmosClient(COSMOSDB_NOSQL_ACCOUNT_ENDPOINT, credential, consistency_level="Strong")

AOAI_client = AzureOpenAI(api_key=OPENAI_API_KEY, azure_endpoint=OPENAI_API_ENDPOINT, api_version=OPENAI_API_VERSION,)

CONTAINER_NAME_DiskANN = "WP-vector-nosql-cont"
CONTAINER_NAME_quantizedFlat = "WP-vector-nosql-cont"

DATABASE_NAME_DiskANN = "WP-vector-nosql-db-DiskANN"
DATABASE_NAME_quantizedFlat = "WP-vector-nosql-db"

dbDiskANN= cosmos_client.get_database_client(DATABASE_NAME_DiskANN)
dbquantizedFlat= cosmos_client.get_database_client(DATABASE_NAME_quantizedFlat)

containerDiskANN = dbDiskANN.get_container_client(CONTAINER_NAME_DiskANN)
containerquantizedFlat = dbquantizedFlat.get_container_client(CONTAINER_NAME_quantizedFlat)


In [7]:
# Simple function to assist with vector search
def vector_search(query, num_results=4, order_by_score=True, DiskANN=True):
    query_embedding = generate_embeddings(query)

    if DiskANN:
        container = containerDiskANN
    else:
        container = containerquantizedFlat
        

    if order_by_score:
        querySQL ='SELECT TOP @num_results c.content, VectorDistance(c.contentVector,@embedding) AS SimilarityScore FROM c ORDER BY VectorDistance(c.contentVector,@embedding)'
    else:
        querySQL ='SELECT TOP @num_results c.content, VectorDistance(c.contentVector,@embedding) AS SimilarityScore FROM c'
    
    results = container.query_items(
            query=querySQL,
            parameters=[{"name": "@embedding", "value": query_embedding},
                        {"name": "@num_results", "value": num_results}],
            enable_cross_partition_query=True,
                populate_index_metrics=True,
                populate_query_metrics=True)

    res = ""
    for result in results: 
        res = res + json.dumps(result["content"], indent=True) + "\r\n"

    print('\n')
    print("RUs: " + container.client_connection.last_response_headers['x-ms-request-charge'].replace(';','\n'))
    floatcost = float(container.client_connection.last_response_headers['x-ms-request-charge'].replace(';','\n')) * 0.000000199
    print("Cost £: " + str('{:f}'.format(floatcost)))
    print(container.client_connection.last_response_headers['x-ms-documentdb-query-metrics'].split(';')[0])
    print('\n')
    #print(containerDiskANN.client_connection.last_response_headers['x-ms-documentdb-query-metrics'].replace(';','\n'))

    return res


In [8]:
#quantizedFlat

query = "Is anyone getting married?"

results = vector_search(query,num_results=2, order_by_score=True, DiskANN=False)

print(results)



RUs: 91.14
Cost £: 0.000018
totalExecutionTimeInMs=21.46


"\"I? I? What of me? Leave me out of the question. I'm not going to get married. What about you? That's what I want to know.\""
"\"Would you marry him?\""



In [9]:
#DiskANN

query = "Is anyone getting married?"

results = vector_search(query,num_results=2, order_by_score=True, DiskANN=True)

print(results)



RUs: 15.38
Cost £: 0.000003
totalExecutionTimeInMs=6.51


"\"I? I? What of me? Leave me out of the question. I'm not going to get married. What about you? That's what I want to know.\""
"\"Would you marry him?\""



#Transactional update example

In [10]:
container = containerDiskANN
#container = containerquantizedFlat
print('injecting new record \n')
        
updaterecord = "I am going to get marry to a dragon no less, called Jeff"

#add a new record to WP-vector-nosql-cont
item = {
    "id": "1000000",
    "category": "Web",
    "content": updaterecord,
    "contentVector": generate_embeddings(updaterecord)
}
container.upsert_item(item)

query = "Is anyone getting married?"

print('Run a query after injecting new record')

results = vector_search(query,3)
print(results)

print('remove the record \n')

container.delete_item("1000000","1000000")

print('Run a query after removing the record')
results = vector_search(query,3)
print(results)


injecting new record 

Run a query after injecting new record


RUs: 19.42
Cost £: 0.000004
totalExecutionTimeInMs=7.41


"\"I? I? What of me? Leave me out of the question. I'm not going to get married. What about you? That's what I want to know.\""
"\"Would you marry him?\""
"The affianced couple, no longer alluding to trees that shed gloom and melancholy upon them, planned the arrangements of a splendid house in Petersburg, paid calls, and prepared everything for a brilliant wedding."

remove the record 

Run a query after removing the record


RUs: 19.43
Cost £: 0.000004
totalExecutionTimeInMs=8.96


"\"I? I? What of me? Leave me out of the question. I'm not going to get married. What about you? That's what I want to know.\""
"\"Would you marry him?\""
"The affianced couple, no longer alluding to trees that shed gloom and melancholy upon them, planned the arrangements of a splendid house in Petersburg, paid calls, and prepared everything for a brilliant wedding."



# Streamlit Demo

In [12]:
! "C:\Python\Python312\Scripts\streamlit" run StreamLitChatCosmosWP.py

^C
